# TelAgentBench-ID Localization and SFT Corpus Builder

This notebook consolidates the two data preparation stages described in Section 3.2 of the manuscript:

1. **Test Suite Localization (`TelAgentBench-ID`)**: Adapts the upstream Korean/English TelAgentBench action subset into everyday Indonesian telecom customer register (`TelkomNusa`, `IndoSatria`, `XLAxiata`) while preserving all 23 BSS API schemas, parameter types, and ground-truth Abstract Syntax Trees (ASTs).
2. **Balanced SFT Corpus Construction (`TelkomNusa-SFT-ID`)**: Builds the 3,510-dialogue ChatML supervised fine-tuning dataset (3,129 train / 381 validation) combining localized Bitext telecom queries, synthetic 23-tool dialogues, multi-turn persona-grounded trajectories, and 850 non-tool `general_support` exchanges, followed by anti-contamination checks (exact deduplication and word 4-gram Jaccard filtering with threshold 0.30).


## 1. Environment Setup and API Client Configuration


In [ ]:
!pip install -q --no-warn-conflicts datasets openai deep-translator pandas tqdm


In [ ]:
import copy
import json
import os
import random
import re
import shutil
import time
import urllib.parse
import urllib.request
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from datasets import load_dataset
from deep_translator import GoogleTranslator
from openai import OpenAI
from tqdm import tqdm

random.seed(42)

# Configure via environment variables or a local .env file before execution.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").strip()
TRANSLATION_MODEL = os.environ.get("TRANSLATION_MODEL", "gpt-4o-mini").strip()

llm_client = None
if OPENAI_API_KEY and not OPENAI_API_KEY.startswith("sk-your"):
    llm_client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    print(f"LLM translation client initialized ({TRANSLATION_MODEL}).")
else:
    print("OPENAI_API_KEY not set; falling back to rule-based + GoogleTranslator pipeline.")


## 2. TelAgentBench-ID Localization Pipeline

Translates user utterances and schema descriptions into natural Indonesian telecom register while locking API names, parameter keys, and ground-truth AST targets.


In [ ]:
class TelecomLocalizer:
    """Batch localizer for Korean/English telecom queries into Indonesian BSS register."""

    def __init__(self, client: Optional[OpenAI], model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model
        self.ko_fallback = GoogleTranslator(source="ko", target="id")
        self.system_prompt = (
            "Anda adalah pakar lokalisasi domain telekomunikasi Indonesia.\n"
            "Terjemahkan daftar kalimat pelanggan ke Bahasa Indonesia alami sesuai konteks layanan seluler.\n"
            "Aturan lokalisasi entitas:\n"
            "1. Operator: SKT/SK Telecom -> TelkomNusa, KT -> IndoSatria, LGU+ -> XLAxiata, T world -> MyTelkomNusa.\n"
            "2. Paket & layanan: baro roaming -> Paket Roaming Passport, refill coupon -> Kupon Extra Kuota, "
            "add-on -> Layanan Tambahan.\n"
            "3. Mata uang & kota: KRW/won -> IDR/Rp (misal 10.000 won -> Rp100.000), Seoul -> Jakarta.\n"
            "Kembalikan HANYA JSON array of string dengan jumlah elemen yang sama persis dengan input."
        )

    @staticmethod
    def apply_glossary_regex(text: str) -> str:
        replacements = [
            (r"\bSKT\b|SK텔레콤|SK Telecom", "TelkomNusa"),
            (r"\bKT\b|케이티", "IndoSatria"),
            (r"\bLGU\+?\b|LG유플러스|LG U\+", "XLAxiata"),
            (r"T world|T월드|T membership|T멤버십", "MyTelkomNusa"),
            (r"baro roaming|baro 로밍|바로그로밍|바로운", "Paket Roaming Passport"),
            (r"리필쿠폰", "Kupon Extra Kuota"),
            (r"부가서비스", "Layanan Tambahan"),
            (r"실시간 요금", "Tagihan Berjalan"),
            (r"확정 요금", "Tagihan Bulanan Resmi"),
            (r"소액 결제", "Potong Pulsa"),
            (r"미납 요금", "Tunggakan Tagihan"),
            (r"잔여 약정", "Sisa Masa Kontrak"),
            (r"가족 데이터", "Kuota Keluarga"),
            (r"서울", "Jakarta"),
            (r"KRW|원(?=[\s\.,\)]|$)", " IDR"),
        ]
        for pattern, repl in replacements:
            text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
        return text

    def translate_batch(self, texts: List[str], src_lang: str = "ko") -> List[str]:
        if not texts:
            return []

        if src_lang == "ko" and not any(re.search(r"[\uac00-\ud7af]", t) for t in texts):
            return [self.apply_glossary_regex(t) for t in texts]

        if self.client is not None:
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": self.system_prompt},
                        {"role": "user", "content": json.dumps(texts, ensure_ascii=False)},
                    ],
                    temperature=0.1,
                    timeout=30,
                )
                content = resp.choices[0].message.content.strip()
                if content.startswith("```"):
                    content = re.sub(r"^```(?:json)?|```$", "", content, flags=re.MULTILINE).strip()
                parsed = json.loads(content)
                if isinstance(parsed, dict):
                    for v in parsed.values():
                        if isinstance(v, list):
                            parsed = v
                            break
                if isinstance(parsed, list) and len(parsed) == len(texts):
                    time.sleep(0.2)
                    return [self.apply_glossary_regex(str(x)) for x in parsed]
            except Exception:
                pass

        results = []
        translator = self.ko_fallback if src_lang == "ko" else GoogleTranslator(source="en", target="id")
        for text in texts:
            try:
                translated = translator.translate(text)
                results.append(self.apply_glossary_regex(translated))
            except Exception:
                results.append(self.apply_glossary_regex(text))
        return results


def load_json_or_jsonl(file_path: str) -> Tuple[Any, bool]:
    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read().strip()
    if not raw:
        return [], False
    try:
        return json.loads(raw), False
    except json.JSONDecodeError:
        lines = [line.strip() for line in raw.splitlines() if line.strip()]
        return [json.loads(line) for line in lines], True


def save_json_or_jsonl(data: Any, out_path: str, is_jsonl: bool) -> None:
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        if is_jsonl and isinstance(data, list):
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        else:
            json.dump(data, f, ensure_ascii=False, indent=2)


def localize_benchmark_file(
    src_path: str,
    dst_path: str,
    localizer: TelecomLocalizer,
    batch_size: int = 15,
) -> None:
    """Translate user prompts and schema descriptions while leaving ground-truth ASTs untouched."""
    if os.path.exists(dst_path):
        try:
            existing, _ = load_json_or_jsonl(dst_path)
            if existing:
                print(f"Skipping existing file: {os.path.basename(dst_path)}")
                return
        except Exception:
            pass

    data, is_jsonl = load_json_or_jsonl(src_path)
    if not data:
        return

    translated_data = copy.deepcopy(data)
    text_targets = []

    if isinstance(translated_data, list):
        for idx, item in enumerate(translated_data):
            if not isinstance(item, dict):
                continue
            if "question" in item and isinstance(item["question"], list):
                for q_idx, q_group in enumerate(item["question"]):
                    if isinstance(q_group, list):
                        for m_idx, msg in enumerate(q_group):
                            if isinstance(msg, dict) and "content" in msg:
                                text_targets.append((idx, ("question", q_idx, m_idx, "content"), msg["content"]))
            if isinstance(item.get("query"), str):
                text_targets.append((idx, ("query",), item["query"]))
            if isinstance(item.get("description"), str):
                text_targets.append((idx, ("description",), item["description"]))
            if isinstance(item.get("parameters"), dict):
                props = item["parameters"].get("properties", {})
                if isinstance(props, dict):
                    for prop_name, prop_val in props.items():
                        if isinstance(prop_val, dict) and "description" in prop_val:
                            text_targets.append(
                                (idx, ("parameters", "properties", prop_name, "description"), prop_val["description"])
                            )

    if text_targets:
        for start in tqdm(range(0, len(text_targets), batch_size), desc=f"Localizing {os.path.basename(src_path)}"):
            batch = text_targets[start : start + batch_size]
            translated_texts = localizer.translate_batch([b[2] for b in batch], src_lang="ko")
            for b_idx, trans_text in enumerate(translated_texts):
                item_idx, field_path, _ = batch[b_idx]
                target = translated_data[item_idx]
                for step in field_path[:-1]:
                    target = target[step]
                target[field_path[-1]] = trans_text

    save_json_or_jsonl(translated_data, dst_path, is_jsonl)


def verify_localized_directory(output_root: str) -> None:
    """Validate JSON syntax and confirm zero residual Hangul characters."""
    checked = 0
    errors = []
    residual_hangul = {}

    for root, _, files in os.walk(output_root):
        for fname in files:
            if not fname.endswith(".json"):
                continue
            checked += 1
            fpath = os.path.join(root, fname)
            try:
                load_json_or_jsonl(fpath)
                with open(fpath, "r", encoding="utf-8") as f:
                    hangul_count = len(re.findall(r"[\uac00-\ud7af]", f.read()))
                    if hangul_count > 0:
                        residual_hangul[fname] = hangul_count
            except Exception as exc:
                errors.append((fname, str(exc)))

    print(f"Verified {checked} JSON files | Parse errors: {len(errors)} | Files with Hangul: {len(residual_hangul)}")
    if errors:
        print("Errors:", errors)
    if residual_hangul:
        print("Residual Hangul:", residual_hangul)


In [ ]:
# Run benchmark localization when raw upstream TelAgentBench files are mounted.
RAW_BENCH_CANDIDATES = [
    "/kaggle/input/telagentbench/dataset_telagentbench_korea",
    "dataset_telagentbench_korea",
]
RAW_INPUT_ROOT = next((p for p in RAW_BENCH_CANDIDATES if os.path.exists(p)), None)
LOCALIZED_BENCH_DIR = os.environ.get("LOCALIZED_BENCH_DIR", "dataset_telagentbench_indonesia")

if RAW_INPUT_ROOT is not None:
    localizer = TelecomLocalizer(llm_client, model=TRANSLATION_MODEL)
    src_possible = os.path.join(RAW_INPUT_ROOT, "TelAgent_Action", "possible_answer")
    dst_possible = os.path.join(LOCALIZED_BENCH_DIR, "TelAgent_Action", "possible_answer")
    if os.path.exists(src_possible):
        os.makedirs(dst_possible, exist_ok=True)
        for fname in os.listdir(src_possible):
            shutil.copy2(os.path.join(src_possible, fname), os.path.join(dst_possible, fname))

    for root, _, files in os.walk(RAW_INPUT_ROOT):
        for fname in files:
            if fname.endswith(".json") and "possible_answer" not in root:
                rel_path = os.path.relpath(os.path.join(root, fname), RAW_INPUT_ROOT)
                localize_benchmark_file(
                    os.path.join(RAW_INPUT_ROOT, rel_path),
                    os.path.join(LOCALIZED_BENCH_DIR, rel_path),
                    localizer,
                )
    verify_localized_directory(LOCALIZED_BENCH_DIR)
else:
    print("Raw upstream benchmark folder not found; skipping Stage 1 localization.")
    print("Pre-localized held-out evaluation files are available on Hugging Face: Rislantrs/TelAgentBench-ID")


## 3. Canonical 23 BSS API Schemas and SFT Corpus Construction

Defines the 23 Indonesian TelkomNusa BSS endpoints (OpenAI/ChatML tool format) and constructs the SFT corpus across:
- Localized Bitext telecom queries (tool-calling + `general_support` no-tool conversations)
- Balanced 23-tool coverage dialogues (family, refill coupons, digital purchases, roaming, auto-payment)
- Multi-turn persona-grounded dialogues (extracting `svc_num` and family attributes from system metadata)


In [ ]:
TELKOMNUSA_BSS_TOOLS = [
    {"type": "function", "function": {"name": "GET__health", "description": "Pemeriksaan status kesehatan sistem layanan (Health Check).", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "GET__rate_add-on-subscriptions", "description": "[Add-on_0001] Menampilkan daftar Layanan Tambahan (Add-on) yang sedang aktif pada nomor pelanggan.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan (contoh: 081234567890)"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "DELETE__rate_add-on-subscriptions", "description": "Membatalkan atau berhenti berlangganan Layanan Tambahan (Add-on).", "parameters": {"type": "object", "properties": {"prodId": {"type": "string", "description": "ID Produk Layanan Tambahan (contoh: AD00000003)"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["prodId", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_content-purchases", "description": "[Bill_0001] Menampilkan rincian biaya penggunaan konten dan layanan digital pada rentang tanggal tertentu.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}, "startDt": {"type": "string", "description": "Tanggal awal pencarian (YYYYMMDD)"}, "endDt": {"type": "string", "description": "Tanggal akhir pencarian (YYYYMMDD)"}}, "required": ["svcMgmtNum", "startDt", "endDt"]}}},
    {"type": "function", "function": {"name": "GET__rate_current-bills", "description": "[Bill_0002] Menampilkan rincian Tagihan Berjalan (real-time) nomor pascabayar bulan ini.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pascabayar pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_current-data-limits", "description": "[Usage_0001] Menampilkan total kuota data, kuota terpakai, dan sisa kuota data saat ini.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__plan", "description": "[Prod_0001] Menampilkan informasi daftar paket tarif, harga langganan, dan produk yang tersedia.", "parameters": {"type": "object", "properties": {"prodId": {"type": "string", "description": "ID Produk Paket Tarif"}, "minProdAmt": {"type": "integer", "description": "Batas minimum harga paket (IDR)"}, "maxProdAmt": {"type": "integer", "description": "Batas maksimum harga paket (IDR)"}, "minProdAge": {"type": "integer", "description": "Batas minimum usia pelanggan"}, "maxProdAge": {"type": "integer", "description": "Batas maksimum usia pelanggan"}}, "required": []}}},
    {"type": "function", "function": {"name": "GET__rate_family_bills", "description": "[Family_0003] Menampilkan tagihan berjalan anggota keluarga yang terdaftar.", "parameters": {"type": "object", "properties": {"familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}}, "required": ["familySvcMgmtNum", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_family_data-usage", "description": "[Family_0001] Menampilkan sisa kuota data anggota keluarga yang terdaftar.", "parameters": {"type": "object", "properties": {"familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}}, "required": ["familySvcMgmtNum", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_family_real-time-bill", "description": "[Family_0008] Menampilkan tagihan berjalan (real-time) nomor anggota keluarga.", "parameters": {"type": "object", "properties": {"familySvcMgmtNum": {"type": "string", "description": "Nomor telepon anggota keluarga"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}}, "required": ["familySvcMgmtNum", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_family_info", "description": "[Family_0009] Menampilkan informasi dan relasi anggota keluarga yang terdaftar.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan utama"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_fixed-bills", "description": "[Bill_0003] Menampilkan Tagihan Bulanan Resmi yang telah dikonfirmasi/terbit.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}, "startDt": {"type": "string", "description": "Tanggal awal periode tagihan (YYYYMMDD)"}, "endDt": {"type": "string", "description": "Tanggal akhir periode tagihan (YYYYMMDD)"}}, "required": ["svcMgmtNum", "startDt", "endDt"]}}},
    {"type": "function", "function": {"name": "GET__rate_mobile-payments", "description": "[Payment_0001] Menampilkan riwayat transaksi Layanan Potong Pulsa pada periode tertentu.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}, "startDt": {"type": "string", "description": "Tanggal awal pencarian (YYYYMMDD)"}, "endDt": {"type": "string", "description": "Tanggal akhir pencarian (YYYYMMDD)"}}, "required": ["svcMgmtNum", "startDt", "endDt"]}}},
    {"type": "function", "function": {"name": "GET__plan_remained-contracts", "description": "[Plan_0003] Menampilkan sisa masa kontrak berlangganan paket atau perangkat HP.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__plan_roaming", "description": "[Roaming_0001] Menampilkan daftar paket roaming internasional yang tersedia untuk negara tujuan.", "parameters": {"type": "object", "properties": {"countryCode": {"type": "string", "description": "Kode Negara (ISO 3166-1 alpha-2, contoh: ID, SG, MY, JP, US, SA)"}, "countryName": {"type": "string", "description": "Nama Negara Tujuan (contoh: Singapura, Jepang, Arab Saudi)"}}, "required": ["countryCode"]}}},
    {"type": "function", "function": {"name": "POST__plan_roaming_subscriptions", "description": "[Roaming_0005] Mendaftarkan atau mengaktifkan paket roaming internasional untuk nomor pelanggan.", "parameters": {"type": "object", "properties": {"roamingProdId": {"type": "string", "description": "ID Produk Paket Roaming (contoh: RM00000007)"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["roamingProdId", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "DELETE__plan_roaming_subscriptions", "description": "[Roaming_0002] Membatalkan atau menonaktifkan paket roaming yang sedang aktif.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_unpaid-bills", "description": "[Bill_0001] Menampilkan rincian Tunggakan Tagihan yang belum dibayar oleh pelanggan.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "POST__rate_unpaid-bills_payment", "description": "[Bill_0002] Melakukan pembayaran untuk melunasi Tunggakan Tagihan pelanggan.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__rate_refill-coupons", "description": "Menampilkan ketersediaan dan status Kupon Extra Kuota (Refill Coupon) pelanggan.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "POST__rate_refill-coupons_use", "description": "Menggunakan atau mengaktifkan Kupon Extra Kuota (jenis: data atau voice).", "parameters": {"type": "object", "properties": {"rfilTrgtltm": {"type": "string", "description": "Jenis kuota yang ditambah (data / voice)"}, "svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["rfilTrgtltm", "svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "DELETE__rate_auto-payment", "description": "Membatalkan atau menonaktifkan layanan pembayaran debit otomatis (Auto-Payment).", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
    {"type": "function", "function": {"name": "GET__info", "description": "[Persona_0001] Menampilkan informasi profil persona dan paket aktif pelanggan.", "parameters": {"type": "object", "properties": {"svcMgmtNum": {"type": "string", "description": "Nomor telepon pelanggan"}}, "required": ["svcMgmtNum"]}}},
]

assert len(TELKOMNUSA_BSS_TOOLS) == 23
print(f"Registered {len(TELKOMNUSA_BSS_TOOLS)} TelkomNusa BSS API schemas.")


In [ ]:
SYSTEM_PROMPT = (
    "Anda adalah asisten AI resmi Customer Care TelkomNusa. Tugas Anda adalah membantu pelanggan "
    "mengecek tagihan, kuota data, paket roaming, kupon extra kuota, paket keluarga, dan layanan "
    "telekomunikasi lainnya secara ramah, sopan, dan akurat. Gunakan fungsi alat (tools) yang "
    "tersedia jika diperlukan."
)


def generate_dummy_phone() -> str:
    prefix = random.choice(["0811", "0812", "0813", "0821", "0822", "0852", "0853", "0881"])
    suffix = "".join(str(random.randint(0, 9)) for _ in range(7))
    return f"{prefix}{suffix}"


def _tool_call_msg(tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "type": "function",
                "function": {"name": tool_name, "arguments": json.dumps(args)},
            }
        ],
    }


def synthesize_bitext_record(row_id: int, indo_query: str, intent: str) -> Dict[str, Any]:
    phone = generate_dummy_phone()
    is_multi_turn = random.random() < 0.25

    if intent in ["dispute_invoice", "invoices"]:
        tool_name = (
            "GET__rate_current-bills"
            if "berjalan" in indo_query.lower() or random.random() < 0.5
            else "GET__rate_unpaid-bills"
        )
        if is_multi_turn:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": indo_query},
                {"role": "assistant", "content": "Halo Kak, boleh dibantu informasikan nomor TelkomNusa yang ingin dicek tagihannya?", "tool_calls": None},
                {"role": "user", "content": f"Nomor saya {phone}"},
                _tool_call_msg(tool_name, {"svcMgmtNum": phone}),
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "billAmt": 145000, "status": "Tercatat", "period": "Juni 2025"})},
                {"role": "assistant", "content": f"Berdasarkan sistem untuk nomor {phone}, tercatat tagihan sebesar Rp145.000 untuk periode Juni 2025."},
            ]
        else:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{indo_query} (Nomor saya {phone})"},
                _tool_call_msg(tool_name, {"svcMgmtNum": phone}),
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "billAmt": 175000, "status": "Belum Lunas", "dueDate": "2025-06-25"})},
                {"role": "assistant", "content": f"Tagihan untuk nomor TelkomNusa {phone} saat ini sebesar Rp175.000 dengan jatuh tempo 25 Juni 2025."},
            ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "billing", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    if intent in ["check_usage", "check_excess_data_charges", "set_usage_limits"]:
        tool_name = "GET__rate_current-data-limits"
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{indo_query} Nomor saya {phone}"},
            _tool_call_msg(tool_name, {"svcMgmtNum": phone}),
            {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "remDataAmt": "18.2 GB", "totOfrDataAmt": "35 GB", "activeUntil": "2025-07-20"})},
            {"role": "assistant", "content": f"Sisa kuota data nomor {phone} saat ini adalah 18.2 GB dari total 35 GB, aktif hingga 20 Juli 2025."},
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "consumption", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    if intent in ["check_mobile_payments", "pay", "payment_methods", "schedule_payments"]:
        if intent == "check_mobile_payments":
            tool_name = "GET__rate_mobile-payments"
            args = {"svcMgmtNum": phone, "startDt": "20250601", "endDt": "20250617"}
            ret = {"resultCode": "0000", "purchaseList": [{"service": "Google Play Store", "amount": 45000, "date": "2025-06-10"}]}
            resp_txt = f"Pada nomor {phone}, tercatat transaksi potong pulsa sebesar Rp45.000 untuk Google Play Store pada 10 Juni 2025."
        else:
            tool_name = "POST__rate_unpaid-bills_payment"
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "paymentStatus": "Sukses", "paidAmt": 150000}
            resp_txt = f"Pembayaran tagihan Rp150.000 untuk nomor {phone} telah berhasil diproses."
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{indo_query} (Nomor: {phone})"},
            _tool_call_msg(tool_name, args),
            {"role": "tool", "name": tool_name, "content": json.dumps(ret)},
            {"role": "assistant", "content": resp_txt},
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "payment", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    if intent in ["activate_roaming", "activate_call_management_services", "deactivate_call_management_services"]:
        if intent == "activate_roaming":
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{indo_query} Saya mau ke Singapura."},
                _tool_call_msg("GET__plan_roaming", {"countryCode": "SG", "countryName": "Singapura"}),
                {"role": "tool", "name": "GET__plan_roaming", "content": json.dumps({"resultCode": "0000", "packages": [{"prodId": "RM00000007", "name": "Paket Roaming Passport Asia 7 Hari (10GB)", "price": 150000}]})},
                {"role": "assistant", "content": "Untuk perjalanan ke Singapura tersedia Paket Roaming Passport Asia 7 Hari (10GB) seharga Rp150.000."},
            ]
        else:
            tool_name = "DELETE__rate_add-on-subscriptions"
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"{indo_query} Nomor saya {phone}"},
                _tool_call_msg(tool_name, {"prodId": "AD00000003", "svcMgmtNum": phone}),
                {"role": "tool", "name": tool_name, "content": json.dumps({"resultCode": "0000", "status": "Sukses Berhenti Berlangganan"})},
                {"role": "assistant", "content": f"Layanan tambahan pada nomor {phone} telah dinonaktifkan."},
            ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "services", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    if intent in ["cancel_plan", "change_plan", "check_cancellation_fee", "sign_up_for_plan"]:
        if "cari" in indo_query.lower() or "rekomendasi" in indo_query.lower():
            tool_name = "GET__plan"
            args = {"minProdAmt": 50000, "maxProdAmt": 100000}
            ret = {"resultCode": "0000", "plans": [{"prodId": "PL001", "name": "TelkomNusa Super Max 50GB", "price": 85000}]}
            resp_txt = "Rekomendasi paket yang sesuai adalah TelkomNusa Super Max 50GB seharga Rp85.000/bulan."
        else:
            tool_name = "GET__plan_remained-contracts"
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "contractStatus": "Aktif", "remainedMonths": 4, "penaltyFee": 0}
            resp_txt = f"Sisa masa kontrak nomor {phone} tercatat 4 bulan lagi tanpa biaya penalti."
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{indo_query} (Nomor: {phone})"},
            _tool_call_msg(tool_name, args),
            {"role": "tool", "name": tool_name, "content": json.dumps(ret)},
            {"role": "assistant", "content": resp_txt},
        ]
        return {"id": f"sft_telko_{row_id:05d}", "category": "subscription", "tools": TELKOMNUSA_BSS_TOOLS, "messages": messages}

    # General support / non-transactional queries (direct natural-language response, no tool call).
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": indo_query},
        {
            "role": "assistant",
            "content": (
                "Mohon maaf atas ketidaknyamanan yang dialami. Laporan kendala Anda telah kami catat "
                "untuk diteruskan ke tim jaringan kami. Silakan mulai ulang perangkat atau hubungi layanan "
                "pelanggan 188 untuk penanganan lanjutan."
            ),
            "tool_calls": None,
        },
    ]
    return {"id": f"sft_telko_{row_id:05d}", "category": "general_support", "tools": [], "messages": messages}


def generate_dedicated_bss_samples(start_id: int) -> List[Dict[str, Any]]:
    """Synthesize dialogues covering all 23 TelkomNusa BSS API endpoints."""
    samples = []
    cur_id = start_id
    family_relations = ["anak", "istri", "suami", "orang tua", "adik", "kakak"]

    for _ in range(150):
        main_phone = generate_dummy_phone()
        fam_phone = generate_dummy_phone()
        rel = random.choice(family_relations)
        sub_case = random.choice(["usage", "bill", "realtime_bill", "info"])

        if sub_case == "usage":
            t_name = "GET__rate_family_data-usage"
            q = f"Tolong cek sisa kuota keluarga nomor {fam_phone} ({rel}) yang terdaftar di nomor utama {main_phone}."
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "remDataAmt": "12.4 GB", "totDataAmt": "20 GB"}
            resp = f"Sisa kuota data keluarga nomor {fam_phone} saat ini adalah 12.4 GB dari 20 GB."
        elif sub_case == "bill":
            t_name = "GET__rate_family_bills"
            q = f"Tolong cek tagihan berjalan {rel} saya nomor {fam_phone}. Nomor utama saya {main_phone}."
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "familyBillAmt": 125000, "period": "Juni 2025"}
            resp = f"Tagihan berjalan nomor keluarga {fam_phone} periode Juni 2025 adalah Rp125.000."
        elif sub_case == "realtime_bill":
            t_name = "GET__rate_family_real-time-bill"
            q = f"Mohon periksa tagihan real-time nomor {fam_phone} dari akun utama {main_phone}."
            args = {"familySvcMgmtNum": fam_phone, "svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "realtimeBill": 98000, "status": "Aktif"}
            resp = f"Tagihan real-time nomor keluarga {fam_phone} saat ini adalah Rp98.000."
        else:
            t_name = "GET__rate_family_info"
            q = f"Tampilkan daftar anggota keluarga yang terdaftar di nomor utama {main_phone}."
            args = {"svcMgmtNum": main_phone}
            ret = {"resultCode": "0000", "familyList": [{"phone": fam_phone, "relation": rel, "status": "Terhubung"}]}
            resp = f"Pada nomor {main_phone}, terdaftar 1 anggota keluarga yaitu {fam_phone} ({rel})."

        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "family_bss",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                _tool_call_msg(t_name, args),
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp},
            ],
        })
        cur_id += 1

    for _ in range(150):
        phone = generate_dummy_phone()
        if random.random() < 0.5:
            t_name = "POST__rate_refill-coupons_use"
            item_type = random.choice(["data", "voice"])
            q = f"Tolong aktifkan Kupon Extra Kuota {item_type} untuk nomor saya {phone}."
            args = {"rfilTrgtltm": item_type, "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "resultMessage": "Sukses", "appliedType": item_type}
            resp = f"Kupon Extra Kuota {item_type} telah berhasil diaktifkan pada nomor {phone}."
        else:
            t_name = "GET__rate_refill-coupons"
            q = f"Apakah nomor {phone} memiliki Kupon Extra Kuota yang bisa digunakan?"
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "availableCoupons": 2, "couponType": ["data", "voice"]}
            resp = f"Nomor {phone} memiliki 2 Kupon Extra Kuota yang tersedia untuk kuota data atau suara."

        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "refill_coupons",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                _tool_call_msg(t_name, args),
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp},
            ],
        })
        cur_id += 1

    for _ in range(150):
        phone = generate_dummy_phone()
        sub_case = random.choice(["content", "auto_pay_del", "fixed_bill", "addon_list"])
        if sub_case == "content":
            t_name = "GET__rate_content-purchases"
            q = f"Tampilkan rincian pembelian konten digital nomor {phone} periode 1 Mei 2025 sampai 31 Mei 2025."
            args = {"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "contentList": [{"name": "Langganan Netflix Premium", "amount": 186000, "date": "20250515"}]}
            resp = f"Pada nomor {phone} terdapat pembelian konten Netflix Premium Rp186.000 pada 15 Mei 2025."
        elif sub_case == "auto_pay_del":
            t_name = "DELETE__rate_auto-payment"
            q = f"Tolong nonaktifkan pembayaran debit otomatis (auto-payment) untuk nomor {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Sukses Dibatalkan"}
            resp = f"Layanan auto-payment untuk nomor {phone} telah berhasil dinonaktifkan."
        elif sub_case == "fixed_bill":
            t_name = "GET__rate_fixed-bills"
            q = f"Periksa Tagihan Bulanan Resmi nomor {phone} dari 01 April 2025 sampai 30 April 2025."
            args = {"startDt": "20250401", "endDt": "20250430", "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "fixedBill": 210000, "status": "Lunas"}
            resp = f"Tagihan Bulanan Resmi nomor {phone} periode April 2025 adalah Rp210.000 (Lunas)."
        else:
            t_name = "GET__rate_add-on-subscriptions"
            q = f"Tampilkan Layanan Tambahan (Add-on) yang aktif pada nomor {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "addOnList": [{"prodId": "AD00000003", "name": "Nada Sambung Pribadi (NSP)", "price": 9000}]}
            resp = f"Pada nomor {phone} terdapat 1 Add-on aktif yaitu Nada Sambung Pribadi seharga Rp9.000/bulan."

        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "digital_transactions",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                _tool_call_msg(t_name, args),
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp},
            ],
        })
        cur_id += 1

    for _ in range(150):
        phone = generate_dummy_phone()
        sub_case = random.choice(["roam_sub", "roam_del", "info_persona", "health"])
        if sub_case == "roam_sub":
            t_name = "POST__plan_roaming_subscriptions"
            prod_id = random.choice(["RM00000007", "RM00000010", "RM00000014"])
            q = f"Aktifkan paket roaming {prod_id} untuk nomor {phone}."
            args = {"roamingProdId": prod_id, "svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Berhasil Terdaftar", "validUntil": "2025-07-01"}
            resp = f"Paket Roaming ({prod_id}) telah diaktifkan untuk nomor {phone}."
        elif sub_case == "roam_del":
            t_name = "DELETE__plan_roaming_subscriptions"
            q = f"Saya sudah kembali ke Indonesia, tolong batalkan paket roaming nomor {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "status": "Paket Roaming Sukses Diberhentikan"}
            resp = f"Paket roaming pada nomor {phone} telah berhasil dinonaktifkan."
        elif sub_case == "info_persona":
            t_name = "GET__info"
            q = f"Tampilkan profil akun pelanggan untuk nomor {phone}."
            args = {"svcMgmtNum": phone}
            ret = {"resultCode": "0000", "customerName": "Ahmad Fauzi", "tier": "TelkomNusa Priority Gold", "status": "Aktif"}
            resp = f"Nomor {phone} terdaftar atas nama Ahmad Fauzi (TelkomNusa Priority Gold, Aktif)."
        else:
            t_name = "GET__health"
            q = "Periksa status kesehatan server dan sistem BSS TelkomNusa saat ini."
            args = {}
            ret = {"resultCode": "0000", "status": "UP", "database": "Connected", "latency": "12ms"}
            resp = "Sistem BSS TelkomNusa berada dalam kondisi UP (latensi 12ms)."

        samples.append({
            "id": f"sft_telko_{cur_id:05d}",
            "category": "roaming_and_ops",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": q},
                _tool_call_msg(t_name, args),
                {"role": "tool", "name": t_name, "content": json.dumps(ret)},
                {"role": "assistant", "content": resp},
            ],
        })
        cur_id += 1

    return samples


def generate_persona_multiturn_samples(start_id: int, count: int = 400) -> List[Dict[str, Any]]:
    """Generate multi-turn dialogues where subscriber attributes live in the system persona."""
    samples = []
    cur_id = start_id
    first_names = ["Budi", "Siti", "Ahmad", "Dewi", "Rizky", "Putri", "Hendra", "Nur", "Agus", "Lestari"]
    last_names = ["Santoso", "Wijaya", "Kusuma", "Pratama", "Saputra", "Hidayat", "Setiawan", "Utami"]
    active_plans = [
        ("PL00000001", "Paket TelkomNusa Halo 5G Unlimited", 150000),
        ("PL00000002", "Paket Super Seru 50GB", 100000),
        ("PL00000003", "Paket Juara Internet 30GB", 75000),
    ]
    addons_pool = [
        ("AD00000001", "Ekstra Kuota Nonton 10GB", 25000),
        ("AD00000002", "Vidio Platinum 30 Hari", 35000),
        ("AD00000003", "Disney+ Hotstar Monthly", 39000),
    ]

    for _ in range(count):
        phone = generate_dummy_phone()
        fam_phone = generate_dummy_phone()
        cust_name = f"{random.choice(first_names)} {random.choice(last_names)}"
        fam_name = f"{random.choice(first_names)} {random.choice(last_names)}"
        birth_year = random.randint(1975, 2004)
        birth_dt = f"{birth_year}{random.randint(1, 12):02d}{random.randint(1, 28):02d}"
        _, plan_name, _ = random.choice(active_plans)
        addon_id, addon_name, addon_fee = random.choice(addons_pool)

        persona = {
            "svc_num": phone,
            "cust_nm": cust_name,
            "birth_dt": birth_dt,
            "gender": random.choice(["Pria", "Wanita"]),
            "active_plan": plan_name,
            "family_members": [{"svc_num": fam_phone, "cust_nm": fam_name, "hubungan": "keluarga"}],
        }
        system_content = (
            "Anda adalah asisten AI resmi Customer Care TelkomNusa. Gunakan informasi persona jika relevan.\n"
            f"Persona penggunanya adalah: {json.dumps(persona, ensure_ascii=False)}\n"
            "Aturan: gunakan svc_num pada persona sebagai svcMgmtNum tanpa meminta ulang ke pelanggan. "
            "Tanggal saat ini adalah 17 Juni 2025."
        )

        scenario = random.choice(["bill_and_addon", "usage_and_payment", "bill_and_content", "roaming_and_plan"])
        if scenario == "bill_and_addon":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Tolong periksa apakah ada tagihan yang belum dibayar pada akun saya."},
                _tool_call_msg("GET__rate_unpaid-bills", {"svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_unpaid-bills", "content": json.dumps({"resultCode": "0000", "unpaidBill": {"hasUnpaid": True, "billAmt": 128500, "dueDate": "20250625"}})},
                {"role": "assistant", "content": f"Pada nomor {phone} terdapat tagihan belum dibayar sebesar Rp128.500 (jatuh tempo 25 Juni 2025)."},
                {"role": "user", "content": "Layanan tambahan apa saja yang sedang aktif di nomor saya?"},
                _tool_call_msg("GET__rate_add-on-subscriptions", {"svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_add-on-subscriptions", "content": json.dumps({"resultCode": "0000", "addOnList": [{"prodId": addon_id, "prodNm": addon_name, "prodAmt": addon_fee}]})},
                {"role": "assistant", "content": f"Add-on aktif saat ini adalah {addon_name} ({addon_id}) seharga Rp{addon_fee:,}/bulan."},
                {"role": "user", "content": f"Tolong batalkan langganan layanan {addon_name} tersebut."},
                _tool_call_msg("DELETE__rate_add-on-subscriptions", {"prodId": addon_id, "svcMgmtNum": phone}),
                {"role": "tool", "name": "DELETE__rate_add-on-subscriptions", "content": json.dumps({"resultCode": "0000", "cancelResult": "Sukses"})},
                {"role": "assistant", "content": f"Langganan {addon_name} pada nomor {phone} telah dibatalkan."},
            ]
        elif scenario == "usage_and_payment":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Tolong cek sisa kuota data internet saya saat ini."},
                _tool_call_msg("GET__rate_current-data-limits", {"svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_current-data-limits", "content": json.dumps({"resultCode": "0000", "totalData": "50 GB", "usedData": "23.4 GB", "remData": "26.6 GB"})},
                {"role": "assistant", "content": f"Sisa kuota data nomor {phone} adalah 26.6 GB dari 50 GB."},
                {"role": "user", "content": "Apakah selama bulan Juni 2025 ini ada transaksi potong pulsa?"},
                _tool_call_msg("GET__rate_mobile-payments", {"startDt": "20250601", "endDt": "20250617", "svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_mobile-payments", "content": json.dumps({"resultCode": "0000", "totalAmount": 35000, "purchaseList": [{"service": "Google Play", "amount": 35000}]})},
                {"role": "assistant", "content": f"Untuk periode 1-17 Juni 2025 pada nomor {phone}, tercatat transaksi Google Play sebesar Rp35.000."},
            ]
        elif scenario == "bill_and_content":
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Berapa biaya pembelian konten digital saya pada bulan Mei 2025?"},
                _tool_call_msg("GET__rate_content-purchases", {"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_content-purchases", "content": json.dumps({"resultCode": "0000", "contentList": [{"contentNm": "Langganan Streaming Film", "purchaseAmt": 54000}]})},
                {"role": "assistant", "content": f"Biaya konten digital nomor {phone} bulan Mei 2025 adalah Rp54.000."},
                {"role": "user", "content": "Periksa juga seluruh tagihan resmi bulan Mei 2025."},
                _tool_call_msg("GET__rate_fixed-bills", {"startDt": "20250501", "endDt": "20250531", "svcMgmtNum": phone}),
                {"role": "tool", "name": "GET__rate_fixed-bills", "content": json.dumps({"resultCode": "0000", "totalBillAmt": 165000})},
                {"role": "assistant", "content": f"Tagihan resmi bulan Mei 2025 untuk nomor {phone} adalah Rp165.000."},
            ]
        else:
            age = 2025 - birth_year
            messages = [
                {"role": "system", "content": system_content},
                {"role": "user", "content": "Saya mau ke Jepang minggu depan, paket roaming apa yang tersedia?"},
                _tool_call_msg("GET__plan_roaming", {"countryCode": "JP", "countryName": "Jepang"}),
                {"role": "tool", "name": "GET__plan_roaming", "content": json.dumps({"resultCode": "0000", "packages": [{"prodId": "RM00000002", "name": "Paket Roaming Asia Pass 7 Hari (10GB)", "price": 175000}]})},
                {"role": "assistant", "content": "Tersedia Paket Roaming Asia Pass 7 Hari (10GB) seharga Rp175.000."},
                {"role": "user", "content": "Rekomendasikan paket internet yang cocok untuk kelompok usia saya."},
                _tool_call_msg("GET__plan", {"minProdAge": max(15, age - 5), "maxProdAge": age + 5}),
                {"role": "tool", "name": "GET__plan", "content": json.dumps({"resultCode": "0000", "products": [{"prodId": "NA00000005", "name": "Paket Juara Muda 50GB", "prodAmt": 90000}]})},
                {"role": "assistant", "content": f"Untuk usia {age} tahun tersedia Paket Juara Muda 50GB seharga Rp90.000/bulan."},
            ]

        samples.append({
            "id": f"sft_telko_persona_{cur_id:05d}",
            "category": "persona_multiturn",
            "tools": TELKOMNUSA_BSS_TOOLS,
            "messages": messages,
        })
        cur_id += 1

    return samples


## 4. Anti-Contamination Audit and Train/Validation Split Export

Audits word 4-gram Jaccard similarity ($\tau = 0.30$) and exact-string isolation between the generated SFT corpus and held-out `TelAgentBench-ID` test queries before writing the 90/10 JSONL split.


In [ ]:
def word_ngrams(text: str, n: int = 4) -> set:
    tokens = re.findall(r"\w+", text.lower())
    if len(tokens) < n:
        return {" ".join(tokens)} if tokens else set()
    return {" ".join(tokens[i : i + n]) for i in range(len(tokens) - n + 1)}


def jaccard_similarity(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def run_contamination_audit(
    sft_records: List[Dict[str, Any]],
    heldout_queries: List[str],
    jaccard_threshold: float = 0.30,
) -> Tuple[List[Dict[str, Any]], int, int]:
    """Filter or flag any SFT record matching held-out queries via exact string or 4-gram Jaccard."""
    heldout_norm = {q.strip().lower() for q in heldout_queries if q.strip()}
    heldout_grams = [word_ngrams(q, n=4) for q in heldout_queries if q.strip()]

    clean_records = []
    exact_hits = 0
    jaccard_hits = 0

    for rec in sft_records:
        user_turns = [m.get("content", "") for m in rec["messages"] if m.get("role") == "user"]
        combined_u = " ".join(user_turns).strip().lower()
        if combined_u in heldout_norm:
            exact_hits += 1
            continue
        u_grams = word_ngrams(combined_u, n=4)
        if any(jaccard_similarity(u_grams, hg) >= jaccard_threshold for hg in heldout_grams):
            jaccard_hits += 1
            continue
        clean_records.append(rec)

    return clean_records, exact_hits, jaccard_hits


# Build SFT dataset (set RUN_SFT_BUILD = True to regenerate from Bitext upstream).
RUN_SFT_BUILD = False

if RUN_SFT_BUILD:
    raw_ds = load_dataset("bitext/Bitext-telco-llm-chatbot-training-dataset", split="train")
    df_raw = pd.DataFrame(raw_ds)
    sampled_rows = [
        group.sample(n=min(len(group), 110), random_state=42)
        for _, group in df_raw.groupby("intent")
    ]
    df_sampled = pd.concat(sampled_rows).sample(frac=1, random_state=42).reset_index(drop=True)

    localizer = TelecomLocalizer(llm_client, model=TRANSLATION_MODEL)
    final_sft_records = []
    batch_size = 25
    instructions = df_sampled["instruction"].tolist()
    intents = df_sampled["intent"].tolist()

    for b_start in tqdm(range(0, len(instructions), batch_size), desc="Localizing Bitext"):
        b_indo = localizer.translate_batch(instructions[b_start : b_start + batch_size], src_lang="en")
        for idx_in_b, q_indo in enumerate(b_indo):
            row_id = b_start + idx_in_b + 1
            final_sft_records.append(synthesize_bitext_record(row_id, q_indo, intents[b_start + idx_in_b]))

    final_sft_records.extend(generate_dedicated_bss_samples(start_id=len(final_sft_records) + 1))
    final_sft_records.extend(generate_persona_multiturn_samples(start_id=len(final_sft_records) + 1, count=400))

    # Optional: load held-out queries from localized benchmark for strict contamination filtering.
    heldout_queries = []
    for fname in ["telco_simple.json", "telco_multi_turn_stateless_base.json", "telco_live_irrelevance.json"]:
        fpath = os.path.join(LOCALIZED_BENCH_DIR, "TelAgent_Action", fname)
        if os.path.exists(fpath):
            items, _ = load_json_or_jsonl(fpath)
            for item in items:
                for grp in item.get("question", []):
                    for m in (grp if isinstance(grp, list) else [grp]):
                        if isinstance(m, dict) and m.get("role") == "user":
                            heldout_queries.append(m.get("content", ""))

    final_sft_records, exact_hits, jaccard_hits = run_contamination_audit(
        final_sft_records, heldout_queries, jaccard_threshold=0.30
    )
    print(f"Contamination audit removed: {exact_hits} exact, {jaccard_hits} 4-gram Jaccard >= 0.30")

    random.shuffle(final_sft_records)
    split_idx = int(len(final_sft_records) * 0.9)
    train_data = final_sft_records[:split_idx]
    val_data = final_sft_records[split_idx:]

    out_dir = "dataset_sft_telko_indonesia"
    os.makedirs(out_dir, exist_ok=True)
    for fname, subset in [("train_sft_telko_indonesia.jsonl", train_data), ("val_sft_telko_indonesia.jsonl", val_data)]:
        with open(os.path.join(out_dir, fname), "w", encoding="utf-8") as f:
            for item in subset:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved Train: {len(train_data)} | Val: {len(val_data)} to {out_dir}")
else:
    print("Set RUN_SFT_BUILD = True to rebuild from Bitext, or load the published release via:")
    print('  load_dataset("Rislantrs/TelkomNusa-SFT-ID")  # 3,129 train / 381 val dialogues')
